In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os

In [ ]:
from openai import OpenAI
openai_client = OpenAI()

In [ ]:
# from groq import Groq
# groq_client = Groq(
#     api_key=os.environ.get("GROQ_API_KEY"),
# )

In [ ]:
groq_client = OpenAI(
    base_url = "https://api.groq.com/openai/v1",
    api_key = os.getenv("GROQ_API_KEY")
)

In [ ]:
from google import genai
from google.genai import types
genai_client = genai.Client(api_key=os.getenv("GENAI_API_KEY"))

In [ ]:
def llm(prompt):
    response = groq_client.responses.create(
        model="llama-3.3-70b-versatile",
        input= prompt
    )
    return response.output_text

In [ ]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'm not aware of a specific course you're referring to. Could you please provide more details or context about the course you're interested in joining? That way, I can better assist you and provide more accurate information.


In [ ]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [ ]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [ ]:
documents = []
url_prefix = "https://datatalks.club/faq"
for course in courses_raw:
    url = f"""{url_prefix}{course['path']}"""
    course_response = requests.get(url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)
    
print(len(documents))

In [ ]:
from minsearch import Index
index = Index(
    text_fields=["section","question","answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [ ]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

In [ ]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [ ]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [ ]:
def build_context(search_results):
    lines= []
    
    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc['question'])
        lines.append("A: " + doc["answer"])
        lines.append("")
        
    return '\n'.join(lines).strip()

In [ ]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context = context
    )
    
    return prompt.strip()

In [ ]:
response = groq_client.responses.create(
    input=prompt,
    model="llama-3.3-70b-versatile"
)

In [ ]:
input_price = .44 / 1000000
output_price = .67 / 1000000

cost = (
    input_price * response.usage.input_tokens +
    output_price * response.usage.output_tokens 
)

print(f"{cost:.5f}$")

In [ ]:
def llm(instructions, user_prompt, model=model):
    message_history = [
        {'role':'system','content': instructions},
        {'role':'user','content':user_prompt}
    ] 
    
    # response = groq_client.responses.create(
    #     model=model,
    #     input=message_history
    # )
    
    response = genai_client.models.generate_content(
        model="gemini-3.5-flash",
        contents=user_prompt,
        config=types.GenerateContentConfig(
            system_instruction=instructions
        )
    )

    return response.text

In [ ]:
def rag(query, model=model):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [ ]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index
documents = load_faq_data()
index = build_index(documents)
rag = RAGBase(index, groq_client)
rag.rag("I just discovered the course. Can I join now?")

In [ ]:
from sqlitesearch import TextSearchIndex
index = TextSearchIndex(
    text_fields= ['question','section','answer'],
    keyword_fields=['course'],
    db_path="FAQ.db"
)
index.search("just discovered the course, can i join now?")

In [ ]:
search_tool = {
    'type' : 'function',
    "name" : 'search',
    "description" : "Search the FAQ database for entries matching the given query.",
    "parameters" : {
        "type" : "object",
        "properties" : {
            "query" : {
                "type" : "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["query"],
        "additionalproperties" : False
    }
}

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

In [ ]:
import json

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

call = response.output[1]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

response = groq_client.responses.create(
    model="llama-3.3-70b-versatile",
    input=messages,
    tools=[search_tool],
)

response.output_text